# 随机过程

> 带结构的随机性。随机游走、马尔可夫链和扩散模型背后的数学。

**类型：** 学习
**语言：** Python
**先修：** Phase 1, Lessons 06-07（概率、贝叶斯）
**时间：** ~75 分钟

## 学习目标

- 模拟 1D 和 2D 随机游走，并验证位移的 sqrt(n) 缩放规律
- 构建马尔可夫链模拟器，并通过特征分解计算它的平稳分布
- 实现 Metropolis-Hastings MCMC 和朗之万动力学，用于从目标分布采样
- 将正向扩散过程连接到布朗运动，并解释反向过程如何生成数据

## 要解决的问题

许多 AI 系统都涉及随时间演化的随机性。不是静态随机性 -- 而是结构化、序列化的随机性，每一步都依赖之前发生的事情。

语言模型一次生成一个 token。每个 token 都依赖之前的上下文。模型输出一个概率分布，从中采样，然后继续。这就是一个随机过程。

扩散模型会一步步向图像中加入噪声，直到它变成纯噪声。然后它们反转这个过程，一步步去噪，直到新的图像出现。正向过程是一条马尔可夫链。反向过程是一条学习到的、向后运行的马尔可夫链。

强化学习智能体在环境中采取动作。每个动作都会以某种概率导致一个新状态。智能体在随机世界里遵循随机策略。整个系统就是一个马尔可夫决策过程。

MCMC 采样 -- 贝叶斯推断的支柱 -- 构造一条马尔可夫链，使它的平稳分布正是你想采样的后验分布。

所有这些都建立在四个基础思想上：
1. 随机游走 -- 最简单的随机过程
2. 马尔可夫链 -- 带转移矩阵的结构化随机性
3. 朗之万动力学 -- 带噪声的梯度下降
4. Metropolis-Hastings -- 从任意分布采样

## 核心概念

### 随机游走

从位置 0 开始。每一步抛一枚公平硬币。正面：向右移动（+1）。反面：向左移动（-1）。

n 步之后，你的位置是 n 个随机 +/-1 值的和。期望位置是 0（这个游走没有偏置）。但离原点的期望距离会按 sqrt(n) 增长。

这有点反直觉。这个游走是公平的 -- 两个方向都没有漂移。但随着时间推移，它会离起点越来越远。n 步后的标准差是 sqrt(n)。

```text
Step 0:  Position = 0
Step 1:  Position = +1 or -1
Step 2:  Position = +2, 0, or -2
...
Step 100: Expected distance from origin ~ 10 (sqrt(100))
Step 10000: Expected distance from origin ~ 100 (sqrt(10000))
```

**在 2D 中**，游走会以相同概率向上、向下、向左或向右移动。离原点的距离同样遵循 sqrt(n) 缩放规律。路径会画出类似分形的图案。

**为什么是 sqrt(n)？** 每一步都是 +1 或 -1，概率相等。n 步后，位置 S_n = X_1 + X_2 + ... + X_n，其中每个 X_i 都是 +/-1。每一步的方差是 1，并且各步相互独立，所以 Var(S_n) = n。标准差 = sqrt(n)。根据中心极限定理，S_n / sqrt(n) 会收敛到标准正态分布。

这种 sqrt(n) 缩放在 ML 中到处出现。SGD 噪声按 1/sqrt(batch_size) 缩放。嵌入维度会按 sqrt(d) 缩放。平方根是独立随机加和的标志。

**与布朗运动的联系。** 取一个步长为 1/sqrt(n)、每单位时间走 n 步的随机游走。当 n 趋于无穷时，这个游走会收敛到布朗运动 B(t) -- 一个连续时间过程，其中 B(t) 服从均值 0、方差 t 的正态分布。

布朗运动是扩散的数学基础。它建模流体中粒子的随机抖动、股票价格的波动，以及 -- 至关重要的 -- 扩散模型中的噪声过程。

**赌徒破产问题。** 一个随机游走者从位置 k 出发，在 0 和 N 处有吸收边界。它先到达 N 而不是 0 的概率是多少？对于公平游走：P(reach N) = k/N。这个结果惊人地简单优雅。它连接到鞅理论 -- 公平随机游走是一个鞅（未来期望值 = 当前值）。

### 马尔可夫链

马尔可夫链是一个系统，它会根据固定概率在状态之间转换。关键性质：下一个状态只依赖当前状态，不依赖历史。

```text
P(X_{t+1} = j | X_t = i, X_{t-1} = ...) = P(X_{t+1} = j | X_t = i)
```

这就是马尔可夫性质。它意味着你可以用一个转移矩阵 P 描述完整动态：

```text
P[i][j] = probability of going from state i to state j
```

P 的每一行加起来都是 1（你必须去到某个地方）。

**示例 -- 天气：**

```text
States: Sunny (0), Rainy (1), Cloudy (2)

P = [[0.7, 0.1, 0.2],    (if sunny: 70% sunny, 10% rainy, 20% cloudy)
     [0.3, 0.4, 0.3],    (if rainy: 30% sunny, 40% rainy, 30% cloudy)
     [0.4, 0.2, 0.4]]    (if cloudy: 40% sunny, 20% rainy, 40% cloudy)
```

从任意状态开始。经过许多次转移后，状态分布会收敛到平稳分布 pi，其中 pi * P = pi。这是 P 的左特征向量，对应特征值 1。

对于天气链，平稳分布可能是 [0.53, 0.18, 0.29] -- 长期来看，无论从什么状态开始，53% 的时间都是晴天。

```mermaid
graph LR
    S["Sunny"] -->|0.7| S
    S -->|0.1| R["Rainy"]
    S -->|0.2| C["Cloudy"]
    R -->|0.3| S
    R -->|0.4| R
    R -->|0.3| C
    C -->|0.4| S
    C -->|0.2| R
    C -->|0.4| C
```

**计算平稳分布。** 有两种方法：

1. **幂方法**：用 P 反复乘以任意初始分布。足够多次迭代后，它会收敛。
2. **特征值方法**：找到 P 的左特征向量，对应特征值 1。这也是 P^T 的特征值 1 对应的特征向量。

两种方法都要求链满足收敛条件。

**收敛条件。** 如果马尔可夫链满足以下条件，它会收敛到唯一的平稳分布：
- **不可约**：每个状态都可以从每个其他状态到达
- **非周期**：链不会以固定周期循环

ML 中遇到的大多数链都满足这两个条件。

**吸收状态。** 如果一个状态一旦进入就永远不会离开（P[i][i] = 1），它就是吸收状态。吸收马尔可夫链可以建模带终止状态的过程 -- 会结束的游戏、流失的客户、命中 end-of-text token 的 token 序列。

**混合时间。** 链需要多少步才会“接近”平稳分布？形式化地说，就是到平稳性的总变差距离降到某个阈值以下所需的步数。快速混合 = 只需要少数步。P 的谱间隙（1 减去第二大特征值）控制混合时间。间隙越大，混合越快。

### 与语言模型的联系

语言模型中的 token 生成近似是一个马尔可夫过程。给定当前上下文，模型会输出下一个 token 的分布。温度控制尖锐程度：

```text
P(token_i) = exp(logit_i / temperature) / sum(exp(logit_j / temperature))
```

- 温度 = 1.0：标准分布
- 温度 < 1.0：更尖锐（更确定）
- 温度 > 1.0：更平坦（更随机）
- 温度 -> 0：argmax（贪心）

Top-k sampling 会截断到概率最高的 k 个 tokens。Top-p（nucleus）sampling 会截断到累计概率超过 p 的最小 token 集合。两者都会修改马尔可夫转移概率。

### 布朗运动

随机游走的连续时间极限。位置 B(t) 有三个性质：
1. B(0) = 0
2. B(t) - B(s) 服从均值 0、方差 t - s 的正态分布（当 t > s）
3. 不重叠区间上的增量相互独立

布朗运动连续但处处不可微 -- 它在每个尺度上都在抖动。在平面中，它的路径分形维数为 2。

在离散仿真中，你可以这样近似布朗运动：

```text
B(t + dt) = B(t) + sqrt(dt) * z,    where z ~ N(0, 1)
```

sqrt(dt) 缩放很重要。它来自应用于随机游走的中心极限定理。

### 朗之万动力学

梯度下降寻找函数的最小值。朗之万动力学寻找与 exp(-U(x)/T) 成正比的概率分布，其中 U 是能量函数，T 是温度。

```text
x_{t+1} = x_t - dt * gradient(U(x_t)) + sqrt(2 * T * dt) * z_t
```

两种力作用在粒子上：
1. **梯度力**（-dt * gradient(U)）：推向低能量区域（像梯度下降）
2. **随机力**（sqrt(2*T*dt) * z）：向随机方向推（探索）

当温度 T = 0 时，这就是纯梯度下降。当温度很高时，它几乎是随机游走。在合适的温度下，粒子会探索能量景观，并在低能量区域中停留更久。

**与扩散模型的联系。** 扩散模型的正向过程是：

```text
x_t = sqrt(alpha_t) * x_{t-1} + sqrt(1 - alpha_t) * noise
```

这是一条马尔可夫链，会逐渐把数据与噪声混合。经过足够多步后，x_T 就是纯高斯噪声。

反向过程 -- 从噪声回到数据 -- 也是一条马尔可夫链，但它的转移概率由神经网络学习。网络学会预测每一步加入的噪声，然后把它减掉。

```mermaid
graph LR
    subgraph "Forward Process (add noise)"
        X0["x_0 (data)"] -->|"+ noise"| X1["x_1"]
        X1 -->|"+ noise"| X2["x_2"]
        X2 -->|"..."| XT["x_T (pure noise)"]
    end
    subgraph "Reverse Process (denoise)"
        XT2["x_T (noise)"] -->|"neural net"| XR2["x_{T-1}"]
        XR2 -->|"neural net"| XR1["x_{T-2}"]
        XR1 -->|"..."| XR0["x_0 (generated data)"]
    end
```

### MCMC：马尔可夫链蒙特卡洛

有时你需要从一个分布 p(x) 中采样，你可以计算它（差一个常数），但无法直接采样。贝叶斯后验是经典例子 -- 你知道似然乘以先验，但归一化常数无法处理。

**Metropolis-Hastings** 会构造一条马尔可夫链，使它的平稳分布是 p(x)：

1. 从某个位置 x 开始
2. 从提议分布 Q(x'|x) 中提出一个新位置 x'
3. 计算接受率：a = p(x') * Q(x|x') / (p(x) * Q(x'|x))
4. 以概率 min(1, a) 接受 x'。否则停留在 x。
5. 重复。

如果 Q 是对称的（例如 Q(x'|x) = Q(x|x') = N(x, sigma^2)），比率会简化为 a = p(x') / p(x)。你只需要概率的比率 -- 归一化常数会抵消。

在温和条件下，这条链保证收敛到 p(x)。但如果提议太小（随机游走）或太大（高拒绝率），收敛会很慢。调提议分布是 MCMC 的艺术。

**为什么有效。** 接受率确保细致平衡：位于 x 并移动到 x' 的概率，等于位于 x' 并移动到 x 的概率。细致平衡蕴含 p(x) 是这条链的平稳分布。所以足够多步之后，样本就来自 p(x)。

**实践注意事项：**
- **预热（Burn-in）**：丢弃前 N 个样本。链需要时间从起点到达平稳分布。
- **抽稀（Thinning）**：每隔 k 个样本保留一个，以降低自相关。
- **多条链（Multiple chains）**：从不同起点运行几条链。如果它们收敛到同一分布，就有了收敛的证据。
- **接受率（Acceptance rate）**：对于 d 维中的高斯提议分布，最优接受率大约是 23%（Roberts & Rosenthal, 2001）。太高表示链几乎不移动。太低表示它什么都拒绝。

### AI 中的随机过程

| 过程 | AI 应用 |
|---------|---------------|
| 随机游走 | RL 中的探索、Node2Vec 嵌入 |
| 马尔可夫链 | 文本生成、MCMC 采样 |
| 布朗运动 | 扩散模型（正向过程） |
| 朗之万动力学 | 基于分数的生成模型、SGLD |
| 马尔可夫决策过程 | 强化学习 |
| Metropolis-Hastings | 贝叶斯推断、后验采样 |

## 动手实现

### 步骤 1：随机游走模拟器


In [ ]:
import numpy as np

def random_walk_1d(n_steps, seed=None):
    rng = np.random.RandomState(seed)
    steps = rng.choice([-1, 1], size=n_steps)
    positions = np.concatenate([[0], np.cumsum(steps)])
    return positions


def random_walk_2d(n_steps, seed=None):
    rng = np.random.RandomState(seed)
    directions = rng.choice(4, size=n_steps)
    dx = np.zeros(n_steps)
    dy = np.zeros(n_steps)
    dx[directions == 0] = 1   # right
    dx[directions == 1] = -1  # left
    dy[directions == 2] = 1   # up
    dy[directions == 3] = -1  # down
    x = np.concatenate([[0], np.cumsum(dx)])
    y = np.concatenate([[0], np.cumsum(dy)])
    return x, y


1D 游走存储累计和。每一步是 +1 或 -1。n 步后，位置就是这些步长的和。方差随 n 线性增长，所以标准差按 sqrt(n) 增长。

### 步骤 2：马尔可夫链


In [ ]:
class MarkovChain:
    def __init__(self, transition_matrix, state_names=None):
        self.P = np.array(transition_matrix, dtype=float)
        self.n_states = len(self.P)
        self.state_names = state_names or [str(i) for i in range(self.n_states)]

    def step(self, current_state, rng=None):
        if rng is None:
            rng = np.random.RandomState()
        probs = self.P[current_state]
        return rng.choice(self.n_states, p=probs)

    def simulate(self, start_state, n_steps, seed=None):
        rng = np.random.RandomState(seed)
        states = [start_state]
        current = start_state
        for _ in range(n_steps):
            current = self.step(current, rng)
            states.append(current)
        return states

    def stationary_distribution(self):
        eigenvalues, eigenvectors = np.linalg.eig(self.P.T)
        idx = np.argmin(np.abs(eigenvalues - 1.0))
        stationary = np.real(eigenvectors[:, idx])
        stationary = stationary / stationary.sum()
        return np.abs(stationary)


平稳分布是 P 的左特征向量，对应特征值 1。我们通过计算 P^T 的特征向量来找到它（转置会把左特征向量变成右特征向量）。

### 步骤 3：朗之万动力学


In [ ]:
def langevin_dynamics(grad_U, x0, dt, temperature, n_steps, seed=None):
    rng = np.random.RandomState(seed)
    x = np.array(x0, dtype=float)
    trajectory = [x.copy()]
    for _ in range(n_steps):
        noise = rng.randn(*x.shape)
        x = x - dt * grad_U(x) + np.sqrt(2 * temperature * dt) * noise
        trajectory.append(x.copy())
    return np.array(trajectory)


梯度会把 x 推向低能量区域。噪声防止它卡住。在平衡时，样本分布与 exp(-U(x)/temperature) 成正比。

### 步骤 4：Metropolis-Hastings


In [ ]:
def metropolis_hastings(target_log_prob, proposal_std, x0, n_samples, seed=None):
    rng = np.random.RandomState(seed)
    x = np.array(x0, dtype=float)
    samples = [x.copy()]
    accepted = 0
    for _ in range(n_samples - 1):
        x_proposed = x + rng.randn(*x.shape) * proposal_std
        log_ratio = target_log_prob(x_proposed) - target_log_prob(x)
        if np.log(rng.rand()) < log_ratio:
            x = x_proposed
            accepted += 1
        samples.append(x.copy())
    acceptance_rate = accepted / (n_samples - 1)
    return np.array(samples), acceptance_rate


这个算法提出一个新点，检查它是否有更高概率（或者以与比率成正比的概率接受），然后重复。为了获得良好的混合，接受率应该在 23-50% 左右。

## 实际使用

实践中，你会使用成熟库来运行这些算法。但理解机制对调试和调参很重要。


In [ ]:
import numpy as np

rng = np.random.RandomState(42)
walk = np.cumsum(rng.choice([-1, 1], size=10000))
print(f"Final position: {walk[-1]}")
print(f"Expected distance: {np.sqrt(10000):.1f}")
print(f"Actual distance: {abs(walk[-1])}")


### numpy 处理转移矩阵


In [ ]:
import numpy as np

P = np.array([[0.7, 0.1, 0.2],
              [0.3, 0.4, 0.3],
              [0.4, 0.2, 0.4]])

distribution = np.array([1.0, 0.0, 0.0])
for _ in range(100):
    distribution = distribution @ P

print(f"Stationary distribution: {np.round(distribution, 4)}")


用 P 反复乘以初始分布。经过足够多次迭代后，无论从哪里开始，它都会收敛到平稳分布。这是寻找主左特征向量的幂方法。

### 与真实框架的联系

- **PyTorch 扩散：** Hugging Face `diffusers` 中的 `DDPMScheduler` 实现了正向和反向马尔可夫链
- **NumPyro / PyMC：** 使用 MCMC（NUTS 采样器，它改进了 Metropolis-Hastings）进行贝叶斯推断
- **Gymnasium (RL)：** 环境 step 函数定义了一个马尔可夫决策过程

### 验证马尔可夫链收敛


In [ ]:
import numpy as np

P = np.array([[0.9, 0.1], [0.3, 0.7]])

eigenvalues = np.linalg.eigvals(P)
spectral_gap = 1 - sorted(np.abs(eigenvalues))[-2]
print(f"Eigenvalues: {eigenvalues}")
print(f"Spectral gap: {spectral_gap:.4f}")
print(f"Approximate mixing time: {1/spectral_gap:.1f} steps")


谱间隙告诉你链多快忘记初始状态。间隙为 0.2，大约意味着 5 步混合。间隙为 0.01，大约意味着 100 步。运行长仿真前一定检查它 -- 混合缓慢的链会浪费计算资源。

## 交付成果

本课产出：
- `outputs/prompt-stochastic-process-advisor.md` -- 一个提示词，帮助识别给定问题适合哪种随机过程框架

## 关联

| 概念 | 出现位置 |
|---------|------------------|
| 随机游走 | Node2Vec 图嵌入、RL 中的探索 |
| 马尔可夫链 | LLMs 的 token 生成、MCMC 采样 |
| 布朗运动 | DDPM 中的正向扩散过程、基于 SDE 的模型 |
| 朗之万动力学 | 基于分数的生成模型、随机梯度朗之万动力学（SGLD） |
| 平稳分布 | MCMC 收敛目标、PageRank |
| Metropolis-Hastings | 贝叶斯后验采样、模拟退火 |
| 温度 | LLM 采样、RL 中的 Boltzmann 探索、模拟退火 |
| 混合时间 | MCMC 的收敛速度、谱间隙分析 |
| 吸收状态 | 序列结束 token、RL 中的终止状态 |
| 细致平衡 | MCMC 采样器的正确性保证 |

扩散模型值得特别关注。DDPM（Ho et al., 2020）定义了一条正向马尔可夫链：

```text
q(x_t | x_{t-1}) = N(x_t; sqrt(1-beta_t) * x_{t-1}, beta_t * I)
```

其中 beta_t 是噪声调度。经过 T 步后，x_T 近似为 N(0, I)。反向过程由一个神经网络参数化，用来预测噪声：

```text
p_theta(x_{t-1} | x_t) = N(x_{t-1}; mu_theta(x_t, t), sigma_t^2 * I)
```

生成过程中的每一步，都是学习到的马尔可夫链的一步。理解马尔可夫链，就能理解扩散模型如何以及为什么能够生成数据。

SGLD（Stochastic Gradient Langevin Dynamics）把小批量梯度下降和朗之万噪声结合起来。你不计算完整梯度，而是使用随机估计，并加入校准过的噪声。随着学习率衰减，SGLD 会从优化转向采样 -- 你几乎免费获得神经网络的近似贝叶斯后验样本。这是从神经网络获得不确定性估计的最简单方法之一。

贯穿这些连接的关键洞见是：随机过程不只是理论工具。它们是现代 AI 系统内部的计算机制。当你调 LLM 的温度时，你是在调整一条马尔可夫链。当你训练扩散模型时，你是在学习反转一个类似布朗运动的过程。当你运行贝叶斯推断时，你是在构造一条会收敛到后验分布的链。

## 练习

1. **模拟 1000 条各 10000 步的随机游走。** 绘制最终位置的分布。验证它近似高斯分布，均值为 0，标准差为 sqrt(10000) = 100。

2. **使用马尔可夫链构建文本生成器。** 在一个小语料库上训练：对每个词，统计到下一个词的转移。构建转移矩阵。通过从链中采样生成新句子。

3. **使用 Metropolis-Hastings 实现模拟退火。** 从高温开始（几乎接受所有内容），然后逐步降温（只接受改进）。用它寻找一个有许多局部最小值的函数的最小值。

4. **比较不同温度下的朗之万动力学。** 从双阱势 U(x) = (x^2 - 1)^2 中采样。低温下，样本聚集在一个势阱中。高温下，它们分散到两个势阱。找到链能在势阱之间混合的临界温度。

5. **实现正向扩散过程。** 从一个 1D 信号（例如正弦波）开始。使用线性噪声调度，在 100 步中逐渐加入噪声。展示信号如何退化成纯噪声。然后实现一个简单去噪器反转这个过程（即使只是简单减去估计噪声的朴素去噪器也可以）。

## 关键术语

| 术语 | 常见说法 | 实际含义 |
|------|----------------|----------------------|
| 随机游走 | “抛硬币式移动” | 一个每一步位置都按随机增量改变的过程 |
| 马尔可夫性质 | “无记忆” | 未来只依赖当前状态，不依赖历史 |
| 转移矩阵 | “概率表” | P[i][j] = 从状态 i 移动到状态 j 的概率 |
| 平稳分布 | “长期平均” | 满足 pi*P = pi 的分布 -- 链的平衡态 |
| 布朗运动 | “随机抖动” | 随机游走的连续时间极限，B(t) ~ N(0, t) |
| 朗之万动力学 | “带噪声的梯度下降” | 结合确定性梯度和随机扰动的更新规则 |
| MCMC | “朝目标走去” | 构造一条马尔可夫链，使它的平稳分布是你想要的分布 |
| Metropolis-Hastings | “提出并接受/拒绝” | 使用接受率确保收敛的 MCMC 算法 |
| 温度 | “随机性旋钮” | 控制探索与利用权衡的参数 |
| 扩散过程 | “噪声进，噪声出” | 正向：逐渐加噪声。反向：逐渐去除噪声。用于生成数据。 |

## 延伸阅读

- **Ho, Jain, Abbeel (2020)** -- "Denoising Diffusion Probabilistic Models." 开启扩散模型热潮的 DDPM 论文。清晰推导了正向和反向马尔可夫链。
- **Song & Ermon (2019)** -- "Generative Modeling by Estimating Gradients of the Data Distribution." 使用朗之万动力学进行采样的基于分数的方法。
- **Roberts & Rosenthal (2004)** -- "General state space Markov chains and MCMC algorithms." 关于 MCMC 何时以及为何有效的理论。
- **Norris (1997)** -- "Markov Chains." 标准教材。覆盖收敛、平稳分布和命中时间。
- **Welling & Teh (2011)** -- "Bayesian Learning via Stochastic Gradient Langevin Dynamics." 把 SGD 与朗之万动力学结合，用于可扩展贝叶斯推断。
